In [1]:
import datetime
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/My Drive/_Project/Files'

Mounted at /content/drive


In [10]:
calls = pd.read_excel(
    f'{folder_path}/Calls (Done).xlsx',
    dtype={'Id': str, 'CONTACTID': str}
)

contacts = pd.read_excel(
    f'{folder_path}/Contacts (Done).xlsx',
    dtype={'Id': str}
)

spend = pd.read_excel(f'{folder_path}/Spend (Done).xlsx')

deals = pd.read_excel(
    f'{folder_path}/Deals (Done).xlsx',
    dtype={'Id': str, 'Contact Name': str}
)

#`Очистка и подготовка данных`

## `calls`

In [11]:
calls.head(3)

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN


In [12]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Id                          95874 non-null  object 
 1   Call Start Time             95874 non-null  object 
 2   Call Owner Name             95874 non-null  object 
 3   CONTACTID                   91941 non-null  object 
 4   Call Type                   95874 non-null  object 
 5   Call Duration (in seconds)  95791 non-null  float64
 6   Call Status                 95874 non-null  object 
 7   Dialled Number              0 non-null      float64
 8   Outgoing Call Status        86875 non-null  object 
 9   Scheduled in CRM            86875 non-null  float64
 10  Tag                         0 non-null      float64
dtypes: float64(4), object(7)
memory usage: 8.0+ MB


Принимаю решение удалить пустые столбцы

In [13]:
calls.drop(['Dialled Number', 'Tag'], axis=1, inplace=True)

### Удаление дубликатов

In [14]:
calls[calls.duplicated(subset=calls.columns.drop('Id'), keep=False)]

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
34,5805028000001140014,06.07.2023 17:15,Alice Johnson,5805028000001129001,Outbound,0.0,Unattended Dialled,Completed,0.0
35,5805028000001167001,06.07.2023 17:15,Alice Johnson,5805028000001129001,Outbound,0.0,Unattended Dialled,Completed,0.0
101,5805028000001372054,08.07.2023 16:43,John Doe,NaN,Missed,0.0,Missed,NaN,NaN
102,5805028000001348077,08.07.2023 16:43,John Doe,NaN,Missed,0.0,Missed,NaN,NaN
254,5805028000001568042,12.07.2023 19:23,Jane Smith,5805028000001552025,Outbound,0.0,Unattended Dialled,Completed,0.0
...,...,...,...,...,...,...,...,...,...
95804,5805028000056832311,21.06.2024 14:17,Yara Edwards,NaN,Outbound,8.0,Attended Dialled,Completed,0.0
95833,5805028000056845313,21.06.2024 14:47,Ulysses Adams,5805028000026041053,Outbound,0.0,Unattended Dialled,Completed,0.0
95834,5805028000056873560,21.06.2024 14:47,Ulysses Adams,5805028000026041053,Outbound,0.0,Unattended Dialled,Completed,0.0
95838,5805028000056834447,21.06.2024 14:55,John Doe,NaN,Missed,0.0,Missed,NaN,NaN


In [15]:
calls.drop_duplicates(subset=calls.columns.drop('Id'), inplace=True)

calls[calls.duplicated(subset=calls.columns.drop('Id'), keep=False)].shape

(0, 9)

### Обработка пропусков

In [16]:
calls_data = pd.DataFrame({
    'NaN': calls.isnull().sum(),
    '% пропусков': calls.isnull().mean() * 100
})
calls_data

,NaN,% пропусков
Id,0,0.000000
Call Start Time,0,0.000000
Call Owner Name,0,0.000000
CONTACTID,3802,4.105078
Call Type,0,0.000000
Call Duration (in seconds),79,0.085298
Call Status,0,0.000000
Outgoing Call Status,8813,9.515532
Scheduled in CRM,8813,9.515532


In [17]:
unique_call_status = calls.loc[
    calls['Call Duration (in seconds)'].isnull(), 'Call Status'].unique()

display(unique_call_status)

array(['Cancelled', 'Overdue', 'Scheduled'], dtype=object)

Статусы Cancelled, Overdue, Scheduled - звонка еще не было или не будет, поэтому в Call Duration (in seconds) NaN заменяю на 0

In [18]:
calls['Call Duration (in seconds)'] = (
    calls['Call Duration (in seconds)']
    .fillna(0)
)

In [19]:
calls[calls[['Outgoing Call Status', 'Scheduled in CRM']].isnull().any(axis=1)]

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN
14,5805028000001030055,05.07.2023 19:17,John Doe,NaN,Missed,0.0,Missed,NaN,NaN
15,5805028000001066016,05.07.2023 19:18,Alice Johnson,NaN,Inbound,13.0,Received,NaN,NaN
16,5805028000001073058,05.07.2023 19:19,John Doe,5805028000000872003,Missed,0.0,Missed,NaN,NaN
17,5805028000001076018,05.07.2023 20:14,John Doe,NaN,Inbound,33.0,Received,NaN,NaN
...,...,...,...,...,...,...,...,...,...
95848,5805028000056834459,21.06.2024 15:03,Derek James,5805028000022524164,Inbound,564.0,Received,NaN,NaN
95853,5805028000056859477,21.06.2024 15:07,Ulysses Adams,NaN,Inbound,407.0,Received,NaN,NaN
95856,5805028000056892448,21.06.2024 15:09,Kevin Parker,5805028000010947071,Missed,0.0,Missed,NaN,NaN
95864,5805028000056873616,21.06.2024 15:24,Kevin Parker,5805028000031509418,Missed,0.0,Missed,NaN,NaN


В Call Type - Inbound, Missed - для входнящего звонка, поэтому в Outgoing Call Status меняю NaN на Inbound, а в планировании 0, так как такие звонки не планируем (потом переведу в bool)

In [21]:
mask = calls['Call Type'].isin(['Inbound', 'Missed'])

calls.loc[mask, 'Outgoing Call Status'] = (
    calls.loc[mask, 'Outgoing Call Status']
    .fillna('Inbound')
)

calls.loc[mask, 'Scheduled in CRM'] = (
    calls.loc[mask, 'Scheduled in CRM']
    .fillna(0)
)

In [22]:
calls['CONTACTID'] = calls['CONTACTID'].fillna('Unknown')

In [23]:
calls_data = pd.DataFrame({
    'NaN': calls.isnull().sum(),
    '% пропусков': calls.isnull().mean() * 100
})
calls_data

,NaN,% пропусков
Id,0,0.0
Call Start Time,0,0.0
Call Owner Name,0,0.0
CONTACTID,0,0.0
Call Type,0,0.0
Call Duration (in seconds),0,0.0
Call Status,0,0.0
Outgoing Call Status,0,0.0
Scheduled in CRM,0,0.0


### Преобразование типов данных

In [24]:
calls.dtypes

,0
Id,object
Call Start Time,object
Call Owner Name,object
CONTACTID,object
Call Type,object
Call Duration (in seconds),float64
Call Status,object
Outgoing Call Status,object
Scheduled in CRM,float64


In [25]:
object_cols = calls.select_dtypes(include=['object']).columns

for col in object_cols:
    print(f"\nВ столбце '{col}' встречаются типы данных и их количество:")
    print(calls[col].map(type).value_counts())


В столбце 'Id' встречаются типы данных и их количество:
Id
<class 'str'>    92617
Name: count, dtype: int64

В столбце 'Call Start Time' встречаются типы данных и их количество:
Call Start Time
<class 'str'>    92617
Name: count, dtype: int64

В столбце 'Call Owner Name' встречаются типы данных и их количество:
Call Owner Name
<class 'str'>    92617
Name: count, dtype: int64

В столбце 'CONTACTID' встречаются типы данных и их количество:
CONTACTID
<class 'str'>    92617
Name: count, dtype: int64

В столбце 'Call Type' встречаются типы данных и их количество:
Call Type
<class 'str'>    92617
Name: count, dtype: int64

В столбце 'Call Status' встречаются типы данных и их количество:
Call Status
<class 'str'>    92617
Name: count, dtype: int64

В столбце 'Outgoing Call Status' встречаются типы данных и их количество:
Outgoing Call Status
<class 'str'>    92617
Name: count, dtype: int64


In [27]:
calls[['Id', 'Call Owner Name', 'CONTACTID', 'Call Status']] = calls[
    ['Id', 'Call Owner Name', 'CONTACTID', 'Call Status']
].astype('string')

calls['Call Start Time'] = (
        pd.to_datetime(calls['Call Start Time'], dayfirst=True, errors="raise")
)

calls[['Call Type', 'Outgoing Call Status']] = (
        calls[['Call Type', 'Outgoing Call Status']].astype('category')
)

calls['Call Duration (in seconds)'] = (
        calls['Call Duration (in seconds)'].astype('int16')

)
calls['Scheduled in CRM'] = calls['Scheduled in CRM'].astype('bool')

In [28]:
calls.dtypes

,0
Id,string[python]
Call Start Time,datetime64[ns]
Call Owner Name,string[python]
CONTACTID,string[python]
Call Type,category
Call Duration (in seconds),int16
Call Status,string[python]
Outgoing Call Status,category
Scheduled in CRM,bool


### Обработка выбросов

In [29]:
calls.describe()

,Call Start Time,Call Duration (in seconds)
count,92617,92617.000000
mean,2024-02-04 19:21:56.171329024,170.184890
min,2023-06-30 08:43:00,0.000000
25%,2023-11-23 17:14:00,4.000000
50%,2024-02-16 16:00:00,9.000000
75%,2024-04-22 14:39:00,107.000000
max,2024-06-21 15:31:00,7625.000000
std,NaN,406.801264


In [30]:
calls[calls['Call Duration (in seconds)'] > 5400]

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
12767,5805028000010429325,2023-10-07 11:23:00,Charlie Davis,5805028000010270024,Outbound,5402,Attended Dialled,Completed,False
28870,5805028000021415097,2023-12-14 15:49:00,Sam Young,5805028000021263687,Outbound,6368,Attended Dialled,Completed,False
29250,5805028000021593632,2023-12-15 19:05:00,Sam Young,5805028000021404645,Outbound,7625,Attended Dialled,Completed,False
56394,5805028000037508725,2024-03-14 19:46:00,John Doe,5805028000021062552,Inbound,6798,Received,Inbound,False
67264,5805028000043303087,2024-04-11 16:00:00,Charlie Davis,5805028000041617254,Outbound,6304,Attended Dialled,Completed,False
78762,5805028000049344069,2024-05-10 17:34:00,Victor Barnes,5805028000047779175,Outbound,7200,Attended Dialled,Completed,False
93729,5805028000055855106,2024-06-17 11:43:00,Eva Kent,5805028000007603031,Outbound,5592,Attended Dialled,Completed,False


### Проверил время звонка более 1,5 часа. На будущее для возможных гипотез - есть менеджеры, которые долго общаются с клиентами, более 1,5 часа. Ошибочных значений не зафиксировал.

## `contacts`

In [32]:
contacts.head(3)

,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17


In [33]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  object
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: object(4)
memory usage: 579.8+ KB


### Удаление дубликатов

In [34]:
contacts[contacts.duplicated(subset=contacts.columns.drop('Id'), keep=False)]

,Id,Contact Owner Name,Created Time,Modified Time
203,5805028000001949074,Bob Brown,17.07.2023 18:40,17.07.2023 18:40
205,5805028000001953081,Bob Brown,17.07.2023 18:40,17.07.2023 18:40
279,5805028000002340967,Bob Brown,18.07.2023 16:53,18.07.2023 16:53
280,5805028000002344049,Bob Brown,18.07.2023 16:53,18.07.2023 16:53
334,5805028000002740077,Bob Brown,21.07.2023 12:26,21.07.2023 12:26
...,...,...,...,...
17864,5805028000054231884,Rachel White,10.06.2024 09:00,10.06.2024 09:33
17865,5805028000054232018,Rachel White,10.06.2024 09:00,10.06.2024 09:33
17869,5805028000054238271,Rachel White,10.06.2024 09:00,10.06.2024 09:33
17870,5805028000054238317,Rachel White,10.06.2024 09:00,10.06.2024 09:33


In [35]:
contacts.drop_duplicates(subset=contacts.columns.drop('Id'), inplace=True)

contacts[contacts.duplicated(subset=contacts.columns[1:])].shape

(0, 4)

### Обработка пропусков

In [36]:
contacts_data = pd.DataFrame({
    'NaN': contacts.isnull().sum(),
    '% пропусков': contacts.isnull().mean() * 100
})
contacts_data

,NaN,% пропусков
Id,0,0.0
Contact Owner Name,0,0.0
Created Time,0,0.0
Modified Time,0,0.0


Пропусков нет

### Преобразование типов данных

In [38]:
contacts.dtypes

,0
Id,object
Contact Owner Name,object
Created Time,object
Modified Time,object


In [39]:
object_cols = contacts.select_dtypes(include=['object']).columns

for col in object_cols:
    print(f"\nВ столбце '{col}' встречаются типы данных и их количество:")
    print(contacts[col].map(type).value_counts())


В столбце 'Id' встречаются типы данных и их количество:
Id
<class 'str'>    18510
Name: count, dtype: int64

В столбце 'Contact Owner Name' встречаются типы данных и их количество:
Contact Owner Name
<class 'str'>     18509
<class 'bool'>        1
Name: count, dtype: int64

В столбце 'Created Time' встречаются типы данных и их количество:
Created Time
<class 'str'>    18510
Name: count, dtype: int64

В столбце 'Modified Time' встречаются типы данных и их количество:
Modified Time
<class 'str'>    18510
Name: count, dtype: int64


In [40]:
row_bool = contacts[contacts['Contact Owner Name'].map(type) == bool]
row_bool

,Id,Contact Owner Name,Created Time,Modified Time
2197,5805028000008772190,False,24.09.2023 09:01,13.10.2023 16:44


In [41]:
calls[calls['CONTACTID'] == '5805028000008772190']

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
9721,5805028000008771348,2023-09-24 11:43:00,George King,5805028000008772190,Outbound,1208,Attended Dialled,Completed,False


In [42]:
contacts.loc[
    contacts['Contact Owner Name'].map(type) == bool,
    'Contact Owner Name'
] = 'George King'

In [43]:
contacts[['Id', 'Contact Owner Name']] = contacts[
    ['Id', 'Contact Owner Name']].astype('string')

contacts['Created Time'] = pd.to_datetime(
    contacts['Created Time'], dayfirst=True, errors="raise"
)

contacts['Modified Time'] = pd.to_datetime(
    contacts['Modified Time'], dayfirst=True, errors="raise"
)

In [44]:
contacts.dtypes

,0
Id,string[python]
Contact Owner Name,string[python]
Created Time,datetime64[ns]
Modified Time,datetime64[ns]


### Обработка выбросов

In [45]:
contacts.describe()

,Created Time,Modified Time
count,18510,18510
mean,2024-01-24 14:25:56.155591680,2024-02-15 09:11:05.273905920
min,2023-06-27 11:28:00,2023-07-06 10:54:00
25%,2023-11-15 16:49:15,2023-12-09 14:51:45
50%,2024-02-01 18:44:30,2024-02-29 01:12:30
75%,2024-04-12 16:15:45,2024-04-26 22:40:45
max,2024-06-21 15:30:00,2024-06-21 15:32:00


Начальные и конечные значения дат визуально не содержат ошибок

## `spend`

In [47]:
spend.head(3)

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN


In [48]:
spend.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20779 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         20779 non-null  datetime64[ns]
 1   Source       20779 non-null  object        
 2   Campaign     14785 non-null  object        
 3   Impressions  20779 non-null  int64         
 4   Spend        20779 non-null  float64       
 5   Clicks       20779 non-null  int64         
 6   AdGroup      13951 non-null  object        
 7   Ad           13951 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.3+ MB


### Удаление дубликатов

In [49]:
spend[spend.duplicated(keep=False)]

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
753,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
755,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
768,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
789,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
841,2023-07-25,Bloggers,NaN,0,0.0,0,NaN,NaN
...,...,...,...,...,...,...,...,...
20746,2024-06-21,Facebook Ads,NaN,0,0.0,0,NaN,NaN
20750,2024-06-21,SMM,NaN,0,0.0,0,NaN,NaN
20764,2024-06-21,Telegram posts,NaN,0,0.0,0,NaN,NaN
20770,2024-06-21,Organic,NaN,0,0.0,0,NaN,NaN


Дубликаты решил не трогать ввиду сложной взаимосвязи данных.

### Обработка пропусков

In [50]:
spend_data = pd.DataFrame({
    'NaN': spend.isnull().sum(),
    '% пропусков': spend.isnull().mean() * 100
})
spend_data

,NaN,% пропусков
Date,0,0.000000
Source,0,0.000000
Campaign,5994,28.846431
Impressions,0,0.000000
Spend,0,0.000000
Clicks,0,0.000000
AdGroup,6828,32.860099
Ad,6828,32.860099


In [51]:
spend['AdGroup'].unique()

array([nan, 'women', 'wide', 'interest_programming', 'recentlymoved',
       'interest_dataanalytics', 'interest_work',
       'interest_programming – Copy', 'interest_dataanalytics – Copy',
       'LAL1', 'b', 'Com_july_1', 'interest_all', 'Com_august',
       'interest_work_WebDev', 'interest_programming_WebDev',
       'promoposts_b', 'retargeting', 'wide_webdesigner',
       'wide_python-developer', 'wide_qa-engineer',
       'interest_python-developer', 'berlin_wide', 'Com_march',
       'accountant_wide'], dtype=object)

In [52]:
filter_spend = spend[spend['AdGroup'].isnull()]

display(filter_spend['Campaign'].unique())

array(['gen_analyst_DE', 'performancemax_eng_DE', nan, 'comp_search_DE',
       'brand_search_eng_DE', 'discovery_DE', 'blog2_DE', 'bbo_DE',
       'discovery_wide1_AT', 'performancemax_wide_AT',
       '1performancemax_wide_PL'], dtype=object)

Фильтр показывает новые уникальные значения, решаю вставить их вместо NaN в AdGroup (удалив 3 последние символа)

In [53]:
spend.loc[spend['AdGroup'].isnull(), 'AdGroup'] = (
    spend.loc[spend['AdGroup'].isnull(), 'Campaign'].str[:-3])

In [55]:
spend_data = pd.DataFrame({
    'NaN': spend.isnull().sum(),
    '% пропусков': spend.isnull().mean() * 100
})
spend_data

,NaN,% пропусков
Date,0,0.000000
Source,0,0.000000
Campaign,5994,28.846431
Impressions,0,0.000000
Spend,0,0.000000
Clicks,0,0.000000
AdGroup,5917,28.475865
Ad,6828,32.860099


Принимаю решение оставить NaN

### Преобразование типов данных

In [56]:
spend.dtypes

,0
Date,datetime64[ns]
Source,object
Campaign,object
Impressions,int64
Spend,float64
Clicks,int64
AdGroup,object
Ad,object


In [58]:
object_cols = spend.select_dtypes(include=['object']).columns

for col in object_cols:
    print(f"\nВ столбце '{col}' встречаются типы данных и их количество:")
    print(spend[col].map(type).value_counts())


В столбце 'Source' встречаются типы данных и их количество:
Source
<class 'str'>    20779
Name: count, dtype: int64

В столбце 'Campaign' встречаются типы данных и их количество:
Campaign
<class 'str'>      14785
<class 'float'>     5994
Name: count, dtype: int64

В столбце 'AdGroup' встречаются типы данных и их количество:
AdGroup
<class 'str'>      14862
<class 'float'>     5917
Name: count, dtype: int64

В столбце 'Ad' встречаются типы данных и их количество:
Ad
<class 'str'>      13951
<class 'float'>     6828
Name: count, dtype: int64


In [59]:
spend['Source'] = spend['Source'].astype('string')

spend[['Impressions', 'Clicks']] = spend[
    ['Impressions', 'Clicks']].astype('int32')

In [ ]:
spend.dtypes

,0
Date,datetime64[ns]
Source,string[python]
Campaign,object
Impressions,int32
Spend,float64
Clicks,int32
AdGroup,object
Ad,object


### Обработка выбросов

In [60]:
spend.describe()

,Date,Impressions,Spend,Clicks
count,20779,20779.000000,20779.000000,20779.000000
mean,2024-01-14 22:32:40.864334080,2458.203475,7.195892,23.990616
min,2023-07-03 00:00:00,0.000000,0.000000,0.000000
25%,2023-10-13 00:00:00,0.000000,0.000000,0.000000
50%,2024-01-27 00:00:00,63.000000,0.580000,1.000000
75%,2024-04-16 00:00:00,709.000000,5.750000,12.000000
max,2024-06-21 00:00:00,431445.000000,774.000000,2415.000000
std,NaN,11442.528075,26.760080,85.245714


In [61]:
display(spend[spend['Impressions'] > 400000])
print()
display(spend[spend['Spend'] > 750])
print()
display(spend[spend['Clicks'] > 2000])

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
2924,2023-09-03,Google Ads,performancemax_eng_DE,431445,236.1,1920,performancemax_eng,NaN


,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
5065,2023-10-10,Bloggers,NaN,9797,773.0,257,NaN,NaN
6841,2023-11-18,Webinar,blog2_DE,16100,773.0,249,blog2,NaN
8487,2023-12-23,Bloggers,NaN,15001,774.0,164,NaN,NaN


,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
926,2023-07-27,Google Ads,performancemax_eng_DE,252824,486.77,2415,performancemax_eng,NaN


Визуально выбросов нет, наблюдается взаимосвязь между 3 столбцами

## `deals`

In [63]:
deals.head(3)

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN


In [64]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21595 entries, 0 to 21594
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   21593 non-null  object 
 1   Deal Owner Name      21564 non-null  object 
 2   Closing Date         14645 non-null  object 
 3   Quality              19340 non-null  object 
 4   Stage                21593 non-null  object 
 5   Lost Reason          16124 non-null  object 
 6   Page                 21593 non-null  object 
 7   Campaign             16067 non-null  object 
 8   SLA                  15533 non-null  object 
 9   Content              14147 non-null  object 
 10  Term                 12454 non-null  object 
 11  Source               21593 non-null  object 
 12  Payment Type         496 non-null    object 
 13  Product              3592 non-null   object 
 14  Education Type       3300 non-null   object 
 15  Created Time         21593 non-null 

После первоначального изучения deals, обрабатываю ошибки

In [65]:
deals['Source'].unique()

array(['Facebook Ads', 'Organic', 'Telegram posts', 'Google Ads',
       'Youtube Ads', 'CRM', 'Webinar', 'SMM', 'Tiktok Ads', 'Bloggers',
       'Partnership', 'Test', 'Offline', nan], dtype=object)

In [66]:
deals.drop(deals[deals['Source'] == 'Test'].index, inplace=True)

In [67]:
deals['Initial Amount Paid'].value_counts()

,count
Initial Amount Paid,
1000,2606
0,876
300,188
500,94
350,82
2000,58
11000,36
200,30
11500,25


In [68]:
deals['Offer Total Amount'].value_counts()

,count
Offer Total Amount,
11000,1842
0,848
11500,394
5000,294
4000,252
3500,133
9000,115
2500,70
2000,62


In [69]:
def clean_money_columns(df, columns):
    for col in columns:
        df[col] = (df[col]
            .replace(r'[€]', '', regex=True)
            .replace(r'\s+', '', regex=True)
            .replace(r'\.', '', regex=True)
            .replace(r',', '.', regex=True)
            .astype(float)
        )
    return df


columns_to_clean = ['Initial Amount Paid', 'Offer Total Amount']
deals = clean_money_columns(deals, columns_to_clean)

In [70]:
error_row = deals[deals['Initial Amount Paid'] > deals['Offer Total Amount']]

display(error_row.head())
display(f"Всего ошибок: {error_row['Initial Amount Paid'].count()}")

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
1279,5805028000053717506,Ben Hall,NaN,C - Low,Call Delayed,NaN,/direct,blog2_DE,01:48:36,NaN,...,Web Developer,Morning,06.06.2024 14:53,6.0,NaN,3000.0,2900.0,5805028000053715483,NaN,NaN
1393,5805028000053561185,Cara Iverson,07.06.2024,D - Non Target,Lost,Non target,/eng,Berlin_DE,05:42:51,b6,...,UX/UI Design,Morning,05.06.2024 08:50,11.0,NaN,11500.0,11000.0,5805028000053534166,Zwickau,NaN
1409,5805028000053462041,Charlie Davis,NaN,D - Non Target,Lost,Gutstein refusal,/eng,performancemax_eng_DE,13:22:11,_{region_name}_,...,UX/UI Design,Morning,04.06.2024 21:24,11.0,NaN,11500.0,11000.0,5805028000053471005,Aschaffenburg,Б2
1440,5805028000053242748,Cara Iverson,NaN,A - High,Waiting For Payment,NaN,/direct,blog2_DE,00:39:03,NaN,...,UX/UI Design,Morning,04.06.2024 12:48,11.0,NaN,11500.0,11000.0,5805028000053244895,Straubing,NaN
1452,5805028000053242571,Quincy Vincent,05.06.2024,D - Non Target,Lost,Non target,/eng,Live_DE,00:18:50,b0,...,UX/UI Design,Morning,04.06.2024 11:13,11.0,NaN,11500.0,11000.0,5805028000053279736,Augsburg,NaN


'Всего ошибок: 58'

In [71]:
condition = deals['Initial Amount Paid'] > deals['Offer Total Amount']

deals.loc[condition, ['Initial Amount Paid', 'Offer Total Amount']] = (
    deals.loc[condition, ['Offer Total Amount', 'Initial Amount Paid']].values
)

display((deals['Initial Amount Paid'] > deals['Offer Total Amount']).sum())

np.int64(0)

In [72]:
counts = deals['Product'].value_counts()
counts

,count
Product,
Digital Marketing,1981
UX/UI Design,1013
Web Developer,573
Find yourself in IT,4
Data Analytics,1


In [73]:
deals[deals['Product'].isin(['Find yourself in IT', 'Data Analytics'])]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
3176,5805028000048377930,Quincy Vincent,07.05.2024,D - Non Target,Lost,Duplicate,/,NaN,NaN,NaN,...,Find yourself in IT,NaN,06.05.2024 20:29,NaN,NaN,1.0,NaN,5805028000048410337,NaN,NaN
6095,5805028000042448604,Julia Nelson,11.05.2024,D - Non Target,Payment Done,Expensive,/page,NaN,NaN,NaN,...,Find yourself in IT,NaN,08.04.2024 17:34,NaN,NaN,1.0,1.0,5805028000005448163,NaN,NaN
10067,5805028000032304003,Charlie Davis,09.05.2024,D - Non Target,Lost,Non target,/event,NaN,NaN,NaN,...,Find yourself in IT,NaN,15.02.2024 19:20,NaN,NaN,0.0,0.0,5805028000012657064,NaN,NaN
16302,5805028000017521117,Julia Nelson,08.05.2024,D - Non Target,Lost,Non target,/workshop,NaN,NaN,NaN,...,Find yourself in IT,NaN,17.11.2023 17:20,NaN,NaN,0.0,0.0,5805028000005448163,NaN,NaN
20914,5805028000003554209,Bob Brown,01.08.2023,D - Non Target,Lost,Went to Rivals,/,NaN,NaN,NaN,...,Data Analytics,NaN,01.08.2023 18:06,NaN,NaN,NaN,6000.0,5805028000003550126,NaN,NaN


Принимаю решение удалить строки с Find yourself in IT и Data Analytics: много пропущенной информации, нет оплат, недостаточное для анализа количество данных

In [74]:
deals = deals[~deals['Product'].isin(['Find yourself in IT', 'Data Analytics'])]

deals[deals['Product'].isin(['Find yourself in IT', 'Data Analytics'])].shape

(0, 23)

### Удаление дубликатов

In [75]:
deals[deals.duplicated(subset=deals.columns.drop('Id'), keep=False)]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
1067,5805028000054269273,John Doe,NaN,NaN,New Lead,NaN,/webinar,NaN,NaN,NaN,...,NaN,NaN,10.06.2024 12:37,NaN,NaN,NaN,NaN,5805028000005448163,NaN,NaN
1069,5805028000054269205,John Doe,NaN,NaN,New Lead,NaN,/webinar,NaN,NaN,NaN,...,NaN,NaN,10.06.2024 12:37,NaN,NaN,NaN,NaN,5805028000005448163,NaN,NaN
11084,5805028000029580838,Charlie Davis,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,NaN,01.02.2024 20:45,NaN,NaN,NaN,NaN,5805028000025816321,NaN,NaN
11085,5805028000029576850,Charlie Davis,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,NaN,...,NaN,NaN,01.02.2024 20:45,NaN,NaN,NaN,NaN,5805028000025816321,NaN,NaN
14754,5805028000020283932,Charlie Davis,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,invitation,...,NaN,NaN,08.12.2023 22:06,NaN,NaN,NaN,NaN,5805028000020421099,NaN,NaN
14755,5805028000020420273,Charlie Davis,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,invitation,...,NaN,NaN,08.12.2023 22:06,NaN,NaN,NaN,NaN,5805028000020421099,NaN,NaN
14807,5805028000020313482,Diana Evans,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,invitation,...,NaN,NaN,08.12.2023 15:44,NaN,NaN,NaN,NaN,5805028000020300454,NaN,NaN
14808,5805028000020284716,Diana Evans,NaN,NaN,Registered on Webinar,NaN,/workshop,NaN,NaN,invitation,...,NaN,NaN,08.12.2023 15:44,NaN,NaN,NaN,NaN,5805028000020300454,NaN,NaN
14938,5805028000020038127,Rachel White,07.12.2023,E - Non Qualified,Lost,Duplicate,/,NaN,NaN,NaN,...,NaN,NaN,07.12.2023 09:03,NaN,NaN,NaN,NaN,5805028000019915422,NaN,NaN
14943,5805028000020038106,Rachel White,07.12.2023,E - Non Qualified,Lost,Duplicate,/,NaN,NaN,NaN,...,NaN,NaN,07.12.2023 09:03,NaN,NaN,NaN,NaN,5805028000019915422,NaN,NaN


In [76]:
deals.drop_duplicates(subset=deals.columns.drop('Id'), inplace=True)

deals[deals.duplicated(subset=deals.columns.drop('Id'), keep=False)].shape

(0, 23)

### Обработка пропусков

In [77]:
deals_data = pd.DataFrame({
    'NaN': deals.isnull().sum(),
    '% пропусков': deals.isnull().mean() * 100
})
deals_data

,NaN,% пропусков
Id,2,0.009336
Deal Owner Name,31,0.144711
Closing Date,6864,32.041826
Quality,2250,10.503221
Stage,2,0.009336
Lost Reason,5392,25.170386
Page,2,0.009336
Campaign,5480,25.581178
SLA,6030,28.148632
Content,7439,34.725983


In [78]:
deals[deals['Id'].isnull()]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
21593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21594,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,#REF!,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
deals.dropna(subset=['Id'], inplace=True)

Принимаю решение: данные, связанные с людьми - заменить NaN на Unknown; NaN в cols заменяю на No_data - буду делать категориальными

In [80]:
deals[['Deal Owner Name', 'Contact Name']] = (
    deals[['Deal Owner Name', 'Contact Name']].fillna('Unknown')
)

cols = ['Quality', 'Lost Reason', 'Payment Type', 'Product', 'Education Type']

deals[cols] = deals[cols].fillna('No_data')

Перевод времени SLA в секунды

In [81]:
display(deals['SLA'].apply(type).value_counts())

display(deals['SLA'].value_counts())

,count
SLA,
<class 'datetime.time'>,13543
<class 'float'>,6028
<class 'datetime.timedelta'>,1849


,count
SLA,
00:10:46,6
00:10:11,6
00:11:09,6
00:17:08,5
00:15:37,5
...,...
00:12:07,1
00:06:58,1
00:07:33,1


In [82]:
def convert_to_seconds(x):
    if pd.isna(x):
        return np.nan
    elif isinstance(x, datetime.time):
        return x.hour * 3600 + x.minute * 60 + x.second
    elif isinstance(x, datetime.timedelta):
        return x.total_seconds()


deals['SLA Seconds'] = deals["SLA"].apply(convert_to_seconds)
deals['SLA Seconds'].value_counts()

,count
SLA Seconds,
646.0,6
611.0,6
669.0,6
1028.0,5
937.0,5
...,...
727.0,1
418.0,1
453.0,1


Удаляю столбец SLA после успешной конвертации

In [83]:
deals.drop(columns=['SLA'], inplace=True)

Принимаю решение Initial Amount Paid и Offer Total Amount очистить от NaN заменой на 0 для удобства последующих расчетов.

In [84]:
deals['Offer Total Amount'] = deals['Offer Total Amount'].fillna(0)
deals['Initial Amount Paid'] = deals['Initial Amount Paid'].fillna(0)

Обработка Level of Deutsch

In [85]:
replace_dict = {
    'в1': 'B1', 'б1': 'B1', 'b1': 'B1', 'B1': 'B1', 'в1-в2': 'B1-B2', 'B2':
    'B2', 'C2': 'C2', 'с1': 'C1', 'Б1': 'B1', 'а2': 'A2', 'а1': 'A1', 'а0':
    'A0', 'б2': 'B2', 'Б2': 'B2', 'В1': 'B1', 'А2': 'A2',
    'B1 будет в феврале 2025': 'B1', 'Detmold, Paulinenstraße 95, 32756':
    'Unknown', 'Сам оценивает на B2, 13 лет живет в Германии': 'B2',
    'в2': 'B2', 'В1-В2': 'B1-B2', 'Б1 ( ждет Б2)': 'B1-B2', 'А2-В1': 'A2-B1',
    'lэкзамен - 6 июля на В1. курсы вечером (но уверенно говорит на B1)': 'B1',
    ('Гражданка Германии уже год в Германии Учит немецкий и в сентябре b1 '
    'через гос-во проходит, а не через ДЖЦ, вечером учится 3 р в неделю с 18 '
    'до 21'): 'B1', '-': 'A0', 'А2 ( Б1 в июне)': 'A2-B1', 'B1 в процессе '
    'обучения': 'B1', 'ЯЗ: нем В1 был экз 03.05 повтор и сейчас ждет '
    'результаты. Технический англ был. А1 сейчас. ОБР: 2 во информационные '
    'и комп сети - инженер системоте': 'B1', 'В1 в сентябре': 'B1', 'Нет':
    'Unknown', 'С1': 'C1', 0: 'A0', 'Ждем B1': 'B1', 'А1 сертиф, но по '
    'факту А2': 'A2', 'a2': 'A2', 'Пока А2, сдает 17 05 B1': 'A2-B1',
    'окончание 13.06 курса на b1': 'B1', 'A1': 'A1', 'b2': 'B2',
    'Thorn-Prikker-Str. 30, Hagen, 58093': 'Unknown', 'В2': 'B2',
    'нулевой уровень, только пошел на курсы.': 'A0', 'ая в1': 'B1',
    'Ждет результат по B1': 'B1', 'А2( ждет итоги Б!)': 'A2-B1',
    'b1 экзамен будет 12 апреля': 'B1', 'b1 (b2 ждет серт)': 'B1-B2',
    'С2': 'C2', 'ждем B1': 'B1', 'Paderborn 33102, Schwabenweg 10':
    'Unknown', 'b1 (B2 ждет серт)': 'B1-B2', 'Ждем B1 со дня на день': 'B1',
    'Б2 ( учит С1)': 'B2-C1', 'B1 еще нет результата': 'B1', '31.05.2024':
    'Unknown', 'Lichtenfelser Straße 25, Untersiemau 96253': 'Unknown',
    'Учиться до сентября на B1': 'B1', 'b1 9ждет экзамен)': 'B1', 'b1+': 'B1',
    'гражданка': 'B2', 'b1 (ждет результат)': 'B1', 'Б1 (учит Б2)': 'B1-B2',
    'б2+': 'B2', 'Гражданин': 'B2', '25 лет живет в Германии': 'C1', 'С1 -ая ,'
    ' Ня -а1': 'C1', 'Ждем результат по B1': 'B1', 'b1 (b2 в июле экзамен)':
    'B1-B2', 'Ждет со дня на день В1': 'B1', 'А2 (В1 с 3 раза не сдала, '
    'бератер видела наши доки)': 'A2-B1', 'b1 (ждет результаты)': 'B1',
    'А2 ( повтор на Б1)': 'A2-B1', 'B1, сдает B2 в апреле': 'B1-B2',
    'ждет сертификат B1': 'B1', 'Б2( 16.02 экзамен С1)': 'B2-C1', 'А1-А2':
    'A1-A2', 'b1 ждет серт на днях на руки': 'B1', 'b1 24 февраля экзамен, '
    'англ b2': 'B1-B2', 'А2 ( скоро екзамен)': 'A2', 'B1 (ждет результаты В2)':
    'B1-B2', 'b1 (b2 15 марта экзамен)': 'B1-B2', 'b2 (с1 экзамен 16 февраля)':
    'B2-C1', 'Б1 ( ждет итог Б2)': 'B1-B2', 'не сдавал, но гражданин': 'B2',
    'Нет сертификатов, но есть С1 англ, неоконченное высшее в ИТ (и еще одно '
    'высшее юридическое) , очень хочет в ИТ, сильно замотивирована именно н':
    'Unknown', 'А2, в процессе Б1': 'A2-B1', 'A0': 'A0',
    'А2(Б1 в марте экз)': 'A2-B1', 'учит A2': 'A2', 'Б1 ( проходит Б2)':
    'B1-B2', 'Б1 ( ждет итог )': 'B1-B2', 'НЯ - В1, АЯ - В1': 'B1',
    'б1 (ждет рез-тат)': 'B1', 'А2(ждет итоги Б1)': 'A2-B1', 'в1-ня , в1-ая':
    'B1', 'ня-0, но англ B2+': 'A0', 'В': 'B1', 'будет B1 в июне': 'B1',
    'А2( включили нем в ангебот)': 'A2', 'а2-в1': 'A2-B1', 'в2-с1': 'B2-C1',
    'курс А2-В1 - сдача в июле, но вечерняя смена инт курсов, настроен '
    'получить гутшайн уже сейчас.': 'A2-B1', 'B1 (B2 должна до конца февраля '
    'получить)': 'B1-B2', 'b1 (b2 экзамен 6 февраля)': 'B1-B2', 'A1-A2':
    'A1-A2', 'Б1( может будет)': 'B1', 'А2 ( в процессе Б1)': 'A2-B1',
    'b1 ждет результаты': 'B1', 'b1 ждет экзамен в феврале': 'B1',
    'В1, может уже В2?': 'B1-B2', 'A2 (идет доучивать В1 - 300 часов; '
    'предположительно до августа)': 'A2-B1', 'не учил': 'A0',
    'Без 5 минут B1 (ждет результаты экзамена)': 'B1', 'а1-а2 , ая свободный':
    'A1-A2', 'b2-c2': 'B2', 'а2, англ B1': 'A2', 'А1': 'A1', 'А2 нем -В2 англ':
    'A2', 'Проходит сейчас B1': 'B1', 'Ждет результат по B1 в феврале': 'B1',
    'Проходит сейчас повторно B1': 'B1', 'b1 экзамен в феврале': 'B1',
    'Учиться на B1 во вторую смену, в первую хочет получить одобрение на '
    'обучение у нас': 'B1', 'Б10Б2': 'B2', 'Б1?': 'B1', 'B1 есть, ждем B2 в '
    'конце месяца': 'B1-B2', 'B1-B2': 'B1-B2', '?': 'Unknown', 'b1 экзамен '
    '26 января': 'B1', 'А0': 'A0', 'а2 (б1 в сер января)': 'A2-B1', 'f2': 'A2',
    'Учиться на B1': 'B1', 'Сдала экзамен на B1, ждет в начале февраля '
    'результат': 'B1', 'Сдавал 8 12 на B1 - ждет результат. 3 01 - аплейт - '
    'получил B1!': 'B1', 'Б1-Б2': 'B1-B2', 'б1 (до июля на В2)': 'B1-B2',
    'А2 ( Б1 март )': 'A2-B1', 'А2 (весной - еще 300 часов В1)': 'A2-B1',
    'В январе будут результаты по экзамену на B1': 'B1',
    'б2 (с1 ждет рез-тат)': 'B2-C1', 'ня-0, ая-B1': 'A0', 'А2-Б1': 'A2-B1',
    'B1 (почти, не сдала чуть) + англ В1': 'B1', 'в1 ждем результаты': 'B1',
    'А2 ( хочет просить совмещать)': 'A2', 'B1 (ждет результаты)': 'B1',
    'А2+': 'A2', 'а2 (сдавала экз В1, но не сдала похоже)': 'A2',
    'в1, идет на в2': 'B1-B2', 'b2-c1': 'B2-C1', 'C1': 'C1', 'b1-b2': 'B1-B2',
    'не учила ( разговорный) сразу пошла работать': 'B2', 'Б1 ( проходит Б2 )':
    'B1-B2', 'a0-a1': 'A0', 'Б1 ( был екзамен ждет итог )': 'B1', 'Б2-С1':
    'B2-C1', 'b1 (учила, но не сдала В2)': 'B1', 'ня а2, ая в1': 'A2-B1',
    'A2 (идет на В1)': 'A2-B1', 'B2-C2': 'B2', 'немецкий - а1-а2, англ b1-b2':
    'A2', 'B2+': 'B2', 'в1, еще нет сертификата': 'B1', 'б1-б2': 'B1-B2',
    'Бй': 'B1', 'ждет результаты по B1 экзамену': 'B1', 'b2 (ждет серт)':
    'B2', 'никакой': 'A0', 'в1 , хочет совмещать с в2': 'B1-B2', 90:
    'Unknown', '.': 'Unknown', 'в1 (уже сдала В2)': 'B2', 'b1 результат '
    'экзамена в феврале': 'B1', 'в1 , экзамен на в2 15 декабря': 'B1-B2',
    'идет на А1': 'A1', 'УТОЧНИТЬ!': 'Unknown', 'B2 (говорит без проблем - '
    'давно здесь)': 'B2', 'B1 (до февраля)': 'B1', 'А2 ( Б2 в процессе)': 'B1',
    'C': 'C1', 'б1 заканчивает': 'B1', 'B1 (B2 экзамен в январе)': 'B1-B2',
    '5 июля 2024 сдает экз на В2': 'B2', 'А2 (заканчив В1 в июне)': 'A2-B1',
    'a2-б1': 'A2-B1', 'В1?': 'B1', 'b1 будет в январе экзамен, готов '
    'совмещать': 'B1', 'b1 (b2 экзамен 2 марта)': 'B1-B2', 'B1 немецкий и '
    'английский Advance': 'B1', 'A': 'A1', 'a2 (b1 экзамен 15 июня)': 'A2-B1',
    'B2 (ждет итог экзамена)': 'B2', 'b1 (b2 не сдал экзамен)': 'B1',
    'В1 (учится на В2 до авг.': 'B1-B2', 'В2 - не сдал': 'B2', 'B2+ (не сдавал,'
    ' но говорит)': 'B2', 'b1 (ждет серт)': 'B1', 'B1 вроде был (18 лет назад '
    'сдавал)': 'B1', 'А2 (сдает B1 - 12 дек) - не сдал!': 'A2',
    'УТОЧНИТЬ': 'Unknown', 'b2 ждет серт': 'B2', 'разговорный из украины, '
    'без сертификата': 'B2', 'Ждет B1': 'B1', 'сдавала А2 в сентябре': 'A2',
    'В1, учится на В2 до няоб 24': 'B1-B2', 'Б1 ( ждет результат Б2)': 'B1-B2',
    'точно уровень не знаю, но говорить могу - учила сама': 'B2', 'А2-В1 учит':
    'A2-B1', 'В1 (учится на В2 уже)': 'B1', 'В январе - В2 сдает': 'B2',
    'b1 должна получить результаты в феврале': 'B1'
}

deals['Level of Deutsch'] = (deals['Level of Deutsch']
    .replace(replace_dict)
    .fillna('Unknown')
)

In [86]:
display(deals['Level of Deutsch'].unique())

display(deals['Level of Deutsch'].value_counts())

array(['Unknown', 'B1', 'A2', 'B1-B2', 'B2', 'C2', 'C1', 'A1', 'A0',
       'A2-B1', 'B2-C1', 'A1-A2'], dtype=object)

,count
Level of Deutsch,
Unknown,20203
B1,750
B2,169
A2,119
B1-B2,63
A2-B1,29
C1,28
A0,24
A1,21


In [87]:
deals['Level of Deutsch'].unique()

array(['Unknown', 'B1', 'A2', 'B1-B2', 'B2', 'C2', 'C1', 'A1', 'A0',
       'A2-B1', 'B2-C1', 'A1-A2'], dtype=object)

In [88]:
replace_take_two = {'B1-B2': 'B2', 'A2-B1': 'B1', 'B2-C1': 'C1', 'A1-A2': 'A2'}
deals['Level of Deutsch'] = deals['Level of Deutsch'].replace(replace_take_two)

In [89]:
mode_values = deals.groupby('Contact Name')['Level of Deutsch'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)
deals['Level of Deutsch'] = deals['Contact Name'].map(mode_values)

Обработка City

In [90]:
mode_values = deals.groupby('Contact Name')['City'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)
deals['City'] = deals['Contact Name'].map(mode_values)

При изучении в Google Sheets увидел значение '-', заменяю на No_data

In [91]:
deals['City'] = deals['City'].replace('-', 'No_data')

In [92]:
deals[['City', 'Level of Deutsch']] = deals[
    ['City', 'Level of Deutsch']
].fillna('No_data')

Принял решение заменить NaN на 0, чтобы перевести в int для экономии памяти, данное значение не повлияет на дальшений анализ

In [93]:
deals[['Course duration', 'Months of study']] = (
    deals[['Course duration', 'Months of study']].fillna(0)
)

In [94]:
deals_data = pd.DataFrame({
    'NaN': deals.isnull().sum(),
    '% пропусков': deals.isnull().mean() * 100
})
deals_data

,NaN,% пропусков
Id,0,0.000000
Deal Owner Name,0,0.000000
Closing Date,6862,32.035481
Quality,0,0.000000
Stage,0,0.000000
Lost Reason,0,0.000000
Page,0,0.000000
Campaign,5478,25.574230
Content,7437,34.719888
Term,9126,42.605042


### Преобразование типов данных

In [95]:
deals.dtypes

,0
Id,object
Deal Owner Name,object
Closing Date,object
Quality,object
Stage,object
Lost Reason,object
Page,object
Campaign,object
Content,object
Term,object


In [96]:
object_cols = deals.select_dtypes(include=['object']).columns

for col in object_cols:
    print(f"\nВ столбце '{col}' встречаются типы данных и их количество:")
    print(deals[col].map(type).value_counts())


В столбце 'Id' встречаются типы данных и их количество:
Id
<class 'str'>    21420
Name: count, dtype: int64

В столбце 'Deal Owner Name' встречаются типы данных и их количество:
Deal Owner Name
<class 'str'>    21420
Name: count, dtype: int64

В столбце 'Closing Date' встречаются типы данных и их количество:
Closing Date
<class 'str'>      14558
<class 'float'>     6862
Name: count, dtype: int64

В столбце 'Quality' встречаются типы данных и их количество:
Quality
<class 'str'>    21420
Name: count, dtype: int64

В столбце 'Stage' встречаются типы данных и их количество:
Stage
<class 'str'>    21420
Name: count, dtype: int64

В столбце 'Lost Reason' встречаются типы данных и их количество:
Lost Reason
<class 'str'>    21420
Name: count, dtype: int64

В столбце 'Page' встречаются типы данных и их количество:
Page
<class 'str'>    21420
Name: count, dtype: int64

В столбце 'Campaign' встречаются типы данных и их количество:
Campaign
<class 'str'>      15942
<class 'float'>     5478
Name

In [97]:
columns_to_string = [
    'Id', 'Deal Owner Name', 'Stage', 'Lost Reason', 'Page',
    'Source', 'Contact Name', 'City', 'Level of Deutsch'
]

deals[columns_to_string] = (
    deals[columns_to_string].astype('string')
)

In [98]:
deals['Closing Date'] = pd.to_datetime(
    deals['Closing Date'],
    dayfirst=True,
    errors='raise'
)

deals['Created Time'] = pd.to_datetime(
    deals['Created Time'],
    dayfirst=True,
    errors="raise"
)

deals[
    ['Quality', 'Payment Type', 'Product', 'Education Type']
] = deals[
    ['Quality', 'Payment Type', 'Product', 'Education Type']
].astype('category')

deals[['Course duration', 'Months of study']] = (
    deals[['Course duration', 'Months of study']]
    .astype('int8')
)

In [99]:
deals.dtypes

,0
Id,string[python]
Deal Owner Name,string[python]
Closing Date,datetime64[ns]
Quality,category
Stage,string[python]
Lost Reason,string[python]
Page,string[python]
Campaign,object
Content,object
Term,object


### Обработка выбросов

In [100]:
deals.describe()

,Closing Date,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,SLA Seconds
count,14558,21420,21420.000000,21420.000000,21420.000000,21420.000000,1.539200e+04
mean,2024-01-27 10:10:53.826074880,2024-01-26 14:27:15.941176576,1.698039,0.212838,178.963632,1386.891643,1.157488e+05
min,2022-10-11 00:00:00,2023-07-03 17:03:00,0.000000,0.000000,0.000000,0.000000,3.000000e+00
25%,2023-11-10 00:00:00,2023-11-18 00:37:45,0.000000,0.000000,0.000000,0.000000,4.338000e+03
50%,2024-02-07 00:00:00,2024-02-04 18:25:00,0.000000,0.000000,0.000000,0.000000,1.968950e+04
75%,2024-04-16 00:00:00,2024-04-13 08:20:15,0.000000,0.000000,0.000000,0.000000,5.606375e+04
max,2024-12-11 00:00:00,2024-06-21 15:30:00,11.000000,11.000000,11000.000000,11500.000000,2.690846e+07
std,NaN,NaN,3.872115,1.203095,690.615112,3485.267594,7.386141e+05


Closing Date и Created Time. Проверяю различие в min.

In [101]:
deals[deals['Closing Date'] == deals['Closing Date'].min()]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA Seconds
18658,5805028000009789201,Julia Nelson,2022-10-11,E - Non Qualified,Lost,Doesn't Answer,eng/digital-marketing,02.07.23wide_DE,b8,wide,...,No_data,2023-10-03 10:36:00,0,0,0.0,0.0,5805028000009784179,No_data,Unknown,2673.0


В Closing Date перепутали год

In [102]:
deals.loc[
    deals['Closing Date'] == pd.Timestamp('2022-10-11'),
    'Closing Date'
] = pd.Timestamp('2023-10-11')

deals[deals['Contact Name'] == '5805028000009784179']

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA Seconds
18658,5805028000009789201,Julia Nelson,2023-10-11,E - Non Qualified,Lost,Doesn't Answer,eng/digital-marketing,02.07.23wide_DE,b8,wide,...,No_data,2023-10-03 10:36:00,0,0,0.0,0.0,5805028000009784179,No_data,Unknown,2673.0


In [106]:
dataframes = {
    "deals": deals,
    "calls": calls,
    "spend": spend,
    "contacts": contacts
}

file_path = f'{folder_path}/dataframes_Yuniov.pkl'

with open(file_path, "wb") as f:
    pickle.dump(dataframes, f)

#`Описательная статистика`

## Сводная статистика

In [107]:
for df, name in zip([calls, contacts, spend, deals],
                    ['Calls', 'Contacts', 'Spend', 'Deals']):
    print(f"\nСводная статистика для {name}:")
    display(df.describe(include='all').fillna('Нет данных'))


Сводная статистика для Calls:


,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
count,92617,92617,92617,92617,92617,92617.0,92617,92617,92617
unique,92617,Нет данных,33,15215,3,Нет данных,11,5,2
top,5805028000000971050,Нет данных,Yara Edwards,Unknown,Outbound,Нет данных,Attended Dialled,Completed,False
freq,1,Нет данных,8532,3802,83804,Нет данных,69522,83725,92481
mean,Нет данных,2024-02-04 19:21:56.171329024,Нет данных,Нет данных,Нет данных,170.18489,Нет данных,Нет данных,Нет данных
min,Нет данных,2023-06-30 08:43:00,Нет данных,Нет данных,Нет данных,0.0,Нет данных,Нет данных,Нет данных
25%,Нет данных,2023-11-23 17:14:00,Нет данных,Нет данных,Нет данных,4.0,Нет данных,Нет данных,Нет данных
50%,Нет данных,2024-02-16 16:00:00,Нет данных,Нет данных,Нет данных,9.0,Нет данных,Нет данных,Нет данных
75%,Нет данных,2024-04-22 14:39:00,Нет данных,Нет данных,Нет данных,107.0,Нет данных,Нет данных,Нет данных
max,Нет данных,2024-06-21 15:31:00,Нет данных,Нет данных,Нет данных,7625.0,Нет данных,Нет данных,Нет данных



Сводная статистика для Contacts:


,Id,Contact Owner Name,Created Time,Modified Time
count,18510,18510,18510,18510
unique,18510,27,Нет данных,Нет данных
top,5805028000000983158,Charlie Davis,Нет данных,Нет данных
freq,1,2018,Нет данных,Нет данных
mean,Нет данных,Нет данных,2024-01-24 14:25:56.155591680,2024-02-15 09:11:05.273905920
min,Нет данных,Нет данных,2023-06-27 11:28:00,2023-07-06 10:54:00
25%,Нет данных,Нет данных,2023-11-15 16:49:15,2023-12-09 14:51:45
50%,Нет данных,Нет данных,2024-02-01 18:44:30,2024-02-29 01:12:30
75%,Нет данных,Нет данных,2024-04-12 16:15:45,2024-04-26 22:40:45
max,Нет данных,Нет данных,2024-06-21 15:30:00,2024-06-21 15:32:00



Сводная статистика для Spend:


,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
count,20779,20779,14785,20779.0,20779.0,20779.0,14862,13951
unique,Нет данных,14,51,Нет данных,Нет данных,Нет данных,34,176
top,Нет данных,Facebook Ads,12.07.2023wide_DE,Нет данных,Нет данных,Нет данных,wide,bloggersvideo9com
freq,Нет данных,9732,2073,Нет данных,Нет данных,Нет данных,5451,714
mean,2024-01-14 22:32:40.864334080,Нет данных,Нет данных,2458.203475,7.195892,23.990616,Нет данных,Нет данных
min,2023-07-03 00:00:00,Нет данных,Нет данных,0.0,0.0,0.0,Нет данных,Нет данных
25%,2023-10-13 00:00:00,Нет данных,Нет данных,0.0,0.0,0.0,Нет данных,Нет данных
50%,2024-01-27 00:00:00,Нет данных,Нет данных,63.0,0.58,1.0,Нет данных,Нет данных
75%,2024-04-16 00:00:00,Нет данных,Нет данных,709.0,5.75,12.0,Нет данных,Нет данных
max,2024-06-21 00:00:00,Нет данных,Нет данных,431445.0,774.0,2415.0,Нет данных,Нет данных



Сводная статистика для Deals:


,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA Seconds
count,21420,21420,14558,21420,21420,21420,21420,15942,13983,12294,...,21420,21420,21420.0,21420.0,21420.0,21420.0,21420,21420,21420,15392.0
unique,21420,28,Нет данных,7,13,22,32,153,182,217,...,3,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,17954,869,8,Нет данных
top,5805028000056864343,Charlie Davis,Нет данных,E - Non Qualified,Lost,No_data,/eng,performancemax_digitalmarkt_ru_DE,_{region_name}_,wide,...,No_data,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Unknown,No_data,Unknown,Нет данных
freq,1,2952,Нет данных,7593,15654,5390,5814,2650,3255,3674,...,18142,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,61,18595,20052,Нет данных
mean,Нет данных,Нет данных,2024-01-27 10:47:00.057700096,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,...,Нет данных,2024-01-26 14:27:15.941176576,1.698039,0.212838,178.963632,1386.891643,Нет данных,Нет данных,Нет данных,115748.762149
min,Нет данных,Нет данных,2023-07-01 00:00:00,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,...,Нет данных,2023-07-03 17:03:00,0.0,0.0,0.0,0.0,Нет данных,Нет данных,Нет данных,3.0
25%,Нет данных,Нет данных,2023-11-10 00:00:00,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,...,Нет данных,2023-11-18 00:37:45,0.0,0.0,0.0,0.0,Нет данных,Нет данных,Нет данных,4338.0
50%,Нет данных,Нет данных,2024-02-07 00:00:00,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,...,Нет данных,2024-02-04 18:25:00,0.0,0.0,0.0,0.0,Нет данных,Нет данных,Нет данных,19689.5
75%,Нет данных,Нет данных,2024-04-16 00:00:00,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,...,Нет данных,2024-04-13 08:20:15,0.0,0.0,0.0,0.0,Нет данных,Нет данных,Нет данных,56063.75
max,Нет данных,Нет данных,2024-12-11 00:00:00,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,Нет данных,...,Нет данных,2024-06-21 15:30:00,11.0,11.0,11000.0,11500.0,Нет данных,Нет данных,Нет данных,26908464.0


## Анализ категориальных полей

In [108]:
dataframes = {
    'calls': calls,
    'contacts': contacts,
    'spend': spend,
    'deals': deals,
}

for table_name, df in dataframes.items():
    print(f"\nАнализ таблицы '{table_name}':")
    category_cols = df.select_dtypes(include='category').columns.tolist()
    if not category_cols:
        print("Нет категориальных столбцов.")
        continue
    for col in category_cols:
        print(f"Анализ столбца '{col}':")
        print(df[col].value_counts())
        print()


Анализ таблицы 'calls':
Анализ столбца 'Call Type':
Call Type
Outbound    83804
Missed       5742
Inbound      3071
Name: count, dtype: int64

Анализ столбца 'Outgoing Call Status':
Outgoing Call Status
Completed    83725
Inbound       8813
Overdue         57
Cancelled       19
Scheduled        3
Name: count, dtype: int64


Анализ таблицы 'contacts':
Нет категориальных столбцов.

Анализ таблицы 'spend':
Нет категориальных столбцов.

Анализ таблицы 'deals':
Анализ столбца 'Quality':
Quality
E - Non Qualified    7593
D - Non Target       6144
C - Low              3445
No_data              2248
B - Medium           1557
A - High              430
F                       3
Name: count, dtype: int64

Анализ столбца 'Payment Type':
Payment Type
No_data               20928
Recurring Payments      349
One Payment             138
Reservation               5
Name: count, dtype: int64

Анализ столбца 'Product':
Product
No_data              17853
Digital Marketing     1981
UX/UI Design          10